# Fill missing query facets via a generative LLM (Claude Sonnet 4.6), few-shot anchored on GEO-Bench

The earlier zero-shot NLI approach disagreed with (even inverted) GEO-Bench's own
`complexity` / `technicality` / `sensitivity` labels (44.7% / 41.2% / 53.8% agreement).
This notebook re-imputes those three facets with a **generative LLM**, reusing the repo's
Claude judge infrastructure (`anthropic.Anthropic()` + `client.messages.create`, the pattern
in `src/attribution.py` / `src/adapters/claude_adapter.py`).

**Method:**
- **Model:** `claude-sonnet-4-6` (the repo's Claude engine; `thesis_config.ENGINES["claude"]`).
- **Few-shot anchored on GEO-Bench:** a small, balanced set of already-labelled queries
  (covering every class of every facet) is put in the system prompt as the labelling standard.
- **Structured output:** the model returns an enum-constrained JSON object
  (`output_config.format`), so every label is valid by construction.
- **Prompt caching:** the rubric + examples sit in a cached system block, so the 250 calls
  reuse the prefix.
- **Fill blanks only:** existing GEO-Bench labels are never overwritten.
- **Same agreement check, target 95%:** re-classify the originally-labelled rows
  (excluding the few-shot exemplars, to avoid leakage) and compare to GEO-Bench.

> Requirements: `pip install anthropic pandas tqdm pyarrow`; `ANTHROPIC_API_KEY` in `.env`.
> Outputs **overwrite** `scaleup_queries_v2_faceted_filled.parquet` and `facet_imputation_log.csv`
> (replacing the discarded NLI fills) and add `facet_agreement_summary.csv`.

## 1. Imports, Claude client, and load the query registry

In [1]:
import json, time
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import sys
from pathlib import Path as _P
_root=_P.cwd()
while _root!=_root.parent and not (_root/"src"/"env.py").exists():
    _root=_root.parent
sys.path.insert(0, str(_root))
from src.env import load_env
load_env()                      # populates ANTHROPIC_API_KEY from .env
import anthropic

MODEL = "claude-sonnet-4-6"     # == thesis_config.ENGINES["claude"]["model"]
client = anthropic.Anthropic(max_retries=5)
print("anthropic", anthropic.__version__, "| model", MODEL)

CANDIDATES = [
    Path("data/scaleup/queries/scaleup_queries_v2_faceted.parquet"),
    Path("../data/scaleup/queries/scaleup_queries_v2_faceted.parquet"),
    Path("scaleup_queries_v2_faceted.parquet"),
]
INPUT_PATH = next((p for p in CANDIDATES if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError("scaleup_queries_v2_faceted.parquet not found; set INPUT_PATH manually.")
INPUT_PATH = INPUT_PATH.resolve(); DATA_DIR = INPUT_PATH.parent
df = pd.read_parquet(INPUT_PATH)
print("input:", INPUT_PATH, "| shape:", df.shape)

anthropic 0.105.2 | model claude-sonnet-4-6
input: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/scaleup_queries_v2_faceted.parquet | shape: (250, 21)


## 2. Facets, missingness, and few-shot exemplars

The exemplars are drawn from the **already-labelled** rows and cover every class of every
facet. They are held out of the agreement check so the validation number isn't inflated by
rows the model saw in its prompt.

In [2]:
LABELSETS = {
    "complexity":   ["simple", "intermediate", "complex"],
    "technicality": ["non-technical", "technical"],
    "sensitivity":  ["non-sensitive", "sensitive"],
}
FACETS = list(LABELSETS)

def is_missing(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except (TypeError, ValueError):
        pass
    return isinstance(v, str) and v.strip() == ""

# Missing-row indices BEFORE filling (the agreement check relies on these).
missing_idx = {col: df.index[df[col].apply(is_missing)].tolist() for col in FACETS}
for col in FACETS:
    print(f"{col:13s} missing: {len(missing_idx[col]):3d} / {len(df)}")
print("TOTAL to fill:", sum(len(v) for v in missing_idx.values()))

# Balanced, deterministic few-shot exemplars from fully-labelled rows.
labelled_full = df[df[FACETS].apply(lambda r: all(not is_missing(r[c]) for c in FACETS), axis=1)]
labelled_full = labelled_full.sort_values("query_id")
EX_PER_CLASS = 2
chosen, chosen_ids = [], set()
for facet in FACETS:
    for label in LABELSETS[facet]:
        cnt = 0
        for _, r in labelled_full.iterrows():
            if r["query_id"] in chosen_ids:
                continue
            if r[facet] == label:
                chosen.append(r); chosen_ids.add(r["query_id"]); cnt += 1
                if cnt >= EX_PER_CLASS:
                    break
EXEMPLARS = pd.DataFrame(chosen)
print("\nfew-shot exemplars:", len(EXEMPLARS))
print(EXEMPLARS[["query_text"] + FACETS].to_string(index=False))

complexity    missing:  31 / 250
technicality  missing:  29 / 250
sensitivity   missing:  51 / 250
TOTAL to fill: 111



few-shot exemplars: 14
                                                                            query_text   complexity  technicality   sensitivity
                                                                          bmi standard       simple non-technical non-sensitive
                                                                      field trip ideas       simple non-technical non-sensitive
                                               why is nitrogen used in plastic welding intermediate     technical non-sensitive
                       How hard would it be for the US to switch to the metric system? intermediate non-technical non-sensitive
          Write an essay discussing the importance of communication in a relationship.      complex non-technical non-sensitive
                                                           How are black holes formed?      complex     technical non-sensitive
                                                            art collaborations o

## 3. Rubric, system prompt, JSON schema, and the classify function

`hypothesis`-free: the model reads the query and the GEO-Bench-anchored examples and returns
an enum-constrained JSON object. The rubric + examples live in a **cached** system block.

In [3]:
RUBRIC = "\n".join([
    "You are an expert query annotator for the GEO-Bench search-query benchmark.",
    "Classify each search query on three facets, matching GEO-Bench's labelling conventions.",
    "",
    "complexity - how much work answering the query takes:",
    "  simple        a direct factual lookup answerable in a sentence",
    "  intermediate  needs some explanation or combining a few facts",
    "  complex       needs detailed analysis, synthesis, or multi-step reasoning",
    "",
    "technicality - the expertise the query assumes:",
    "  non-technical  an everyday, general-audience question",
    "  technical      requires specialist or domain expertise (science, engineering, medicine, law, finance, IT)",
    "",
    "sensitivity - whether the topic is sensitive:",
    "  non-sensitive  general knowledge",
    "  sensitive      involves health, finance, legal, political, safety, or personal matters",
    "",
    "Treat the labelled examples below as the standard for how the labels are applied.",
])

def _ex_line(r):
    labels = json.dumps({c: r[c] for c in FACETS})
    return "Query: " + str(r["query_text"]) + "\n" + labels

FEWSHOT = "\n\n".join(_ex_line(r) for _, r in EXEMPLARS.iterrows())
SYSTEM_TEXT = RUBRIC + "\n\nEXAMPLES\n" + FEWSHOT + "\n\nClassify the user's query."
SYSTEM_BLOCKS = [{"type": "text", "text": SYSTEM_TEXT, "cache_control": {"type": "ephemeral"}}]

SCHEMA = {
    "type": "object",
    "properties": {c: {"type": "string", "enum": LABELSETS[c]} for c in FACETS},
    "required": FACETS,
    "additionalProperties": False,
}

def classify(qtext):
    resp = client.messages.create(
        model=MODEL, max_tokens=256,
        system=SYSTEM_BLOCKS,
        messages=[{"role": "user", "content": "Query: " + str(qtext)}],
        output_config={"format": {"type": "json_schema", "schema": SCHEMA}},
    )
    txt = next(b.text for b in resp.content if getattr(b, "type", "") == "text")
    data = json.loads(txt)
    out = {c: data[c] for c in FACETS}
    for c in FACETS:
        if out[c] not in LABELSETS[c]:
            raise ValueError("bad label " + c + ":" + str(out[c]))
    u = resp.usage
    return out, (u.input_tokens, u.output_tokens, getattr(u, "cache_read_input_tokens", 0) or 0)

def classify_safe(qid, qtext, tries=4):
    for k in range(tries):
        try:
            out, u = classify(qtext); return qid, out, u
        except Exception:
            time.sleep(1.5 * (k + 1))
    return qid, None, (0, 0, 0)

print("system prompt chars:", len(SYSTEM_TEXT))

system prompt chars: 2994


## 4. Classify all 250 queries

One call per query. A single warm-up call writes the prompt cache; the rest run concurrently
and read it. The SDK auto-retries 429/5xx; `classify_safe` adds JSON-validation retries.

In [4]:
_ = classify(df.iloc[0]["query_text"])      # warm the prompt cache

rows = list(df[["query_id", "query_text"]].itertuples(index=False, name=None))
preds, usage = {}, []
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(classify_safe, qid, qt) for qid, qt in rows]
    for f in tqdm(as_completed(futs), total=len(futs), desc="classify"):
        qid, out, u = f.result()
        usage.append(u)
        if out is not None:
            preds[qid] = out

fails = [qid for qid, _ in rows if qid not in preds]
print("classified:", len(preds), "/", len(rows), "| failures:", len(fails))
for qid in fails:                            # sequential retry for stragglers
    qt = df.loc[df.query_id == qid, "query_text"].iloc[0]
    _, out, u = classify_safe(qid, qt, tries=6)
    if out is not None:
        preds[qid] = out; usage.append(u)
print("after retry:", len(preds), "/", len(rows))
assert len(preds) == len(rows), "some queries never classified"

classify:   0%|          | 0/250 [00:00<?, ?it/s]

classified: 250 / 250 | failures: 0
after retry: 250 / 250


## 5. Fill missing values (blanks only)

In [5]:
impute_log = []
for col in FACETS:
    for i in missing_idx[col]:
        qid = df.at[i, "query_id"]
        assert is_missing(df.at[i, col]), "refusing to overwrite an existing label"
        val = preds[qid][col]
        df.at[i, col] = val
        impute_log.append({
            "query_id": qid, "query_text": df.at[i, "query_text"],
            "column_name": col, "imputed_value": val, "model": MODEL,
        })
print("filled:", len(impute_log), "cells")
pd.DataFrame(impute_log).head(10)

filled: 111 cells


,query_id,query_text,column_name,imputed_value,model
0,gb_c30e3946381108e2,'Businesses owned by responsible and organized...,complexity,simple,claude-sonnet-4-6
1,gb_1a90b9dd7be2bf64,login into onedrive,complexity,simple,claude-sonnet-4-6
2,gb_8b26c2d8d5387ac3,wakehealth.edu,complexity,simple,claude-sonnet-4-6
3,gb_157a7eb6b5972399,my service canada account log in,complexity,simple,claude-sonnet-4-6
4,gb_e78889de403c8eaa,http://www.rewardclub.me/,complexity,simple,claude-sonnet-4-6
5,gb_5b95c1501647091c,account/microsoft/login,complexity,simple,claude-sonnet-4-6
6,gb_4ee9cc1110279ca5,prelicensetraining login,complexity,simple,claude-sonnet-4-6
7,gb_c1ee9aa76d2cfa68,msnbc live streaming free online tv,complexity,simple,claude-sonnet-4-6
8,gb_3dfd19034e427687,free online auto repair estimates,complexity,simple,claude-sonnet-4-6
9,gb_780045e98ad742c0,grade to percentage calculator,complexity,simple,claude-sonnet-4-6


## 6. Consistency check vs. GEO-Bench labels (target 95%)

Re-classify the **originally-labelled** rows, excluding the few-shot exemplars, and compare to
GEO-Bench. Validation only — nothing is overwritten.

In [6]:
EX_IDS = set(EXEMPLARS["query_id"])
agg, dis = [], []
for col in FACETS:
    missing_set = set(missing_idx[col])
    idxs = [i for i in df.index
            if i not in missing_set                       # originally labelled
            and df.at[i, "query_id"] not in EX_IDS         # not a few-shot exemplar
            and df.at[i, "query_id"] in preds]
    agree = 0
    for i in idxs:
        pred = preds[df.at[i, "query_id"]][col]
        exist = df.at[i, col]
        if pred == exist:
            agree += 1
        else:
            dis.append({"column_name": col, "query_text": df.at[i, "query_text"],
                        "existing": exist, "predicted": pred})
    n = len(idxs)
    pct = 100 * agree / n if n else float("nan")
    agg.append({"column": col, "checked": n, "agree": agree,
                "agreement_pct": round(pct, 1), "meets_95pct": bool(pct >= 95)})
    flag = "PASS >=95%" if pct >= 95 else "below 95%"
    print(f"{col:13s} agreement {agree}/{n} = {pct:.1f}%   {flag}")

agg_df = pd.DataFrame(agg)
print("\n=== AGREEMENT SUMMARY (target 95%) ===")
display(agg_df)
print("=== SAMPLE DISAGREEMENTS (max 15) ===")
display(pd.DataFrame(dis).head(15))

complexity    agreement 154/205 = 75.1%   below 95%
technicality  agreement 186/207 = 89.9%   below 95%
sensitivity   agreement 155/185 = 83.8%   below 95%

=== AGREEMENT SUMMARY (target 95%) ===


,column,checked,agree,agreement_pct,meets_95pct
0,complexity,205,154,75.1,False
1,technicality,207,186,89.9,False
2,sensitivity,185,155,83.8,False


=== SAMPLE DISAGREEMENTS (max 15) ===


,column_name,query_text,existing,predicted
0,complexity,where does the sound come from when you crack ...,simple,intermediate
1,complexity,"If ""Thou shalt not kill"" is one of the ten com...",intermediate,complex
2,complexity,Why do things far away in the distance like hi...,simple,intermediate
3,complexity,I have a glass that I put nothing but water in...,simple,intermediate
4,complexity,"how come fresh food such as ketchup, mayonnais...",simple,intermediate
5,complexity,Was the destruction of large quantities of con...,intermediate,complex
6,complexity,"What is, or should be, 'global history'?",intermediate,complex
7,complexity,Are there any questions that should be beyond ...,intermediate,complex
8,complexity,Does a religion have to be old?,intermediate,simple
9,complexity,Write the history of a colour.,intermediate,complex


## 7. Summary, cost, and save

In [7]:
for col in FACETS:
    print("---", col, "---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

remaining = {col: int(df[col].apply(is_missing).sum()) for col in FACETS}
print("remaining missing:", remaining)
assert all(v == 0 for v in remaining.values()), "still missing values!"

OUT_PARQUET = DATA_DIR / "scaleup_queries_v2_faceted_filled.parquet"   # overwrites NLI version
OUT_LOG     = DATA_DIR / "facet_imputation_log.csv"
OUT_AGREE   = DATA_DIR / "facet_agreement_summary.csv"
df.to_parquet(OUT_PARQUET, index=False)
pd.DataFrame(impute_log).to_csv(OUT_LOG, index=False)
agg_df.to_csv(OUT_AGREE, index=False)

ti = sum(u[0] for u in usage); to = sum(u[1] for u in usage); tc = sum(u[2] for u in usage)
cost = (ti * 3.0 + tc * 0.30 + to * 15.0) / 1e6     # claude pricing $3/$15 per MTok, cache read ~0.1x
print("saved:", OUT_PARQUET)
print("saved:", OUT_LOG, f"({len(impute_log)} rows)")
print("saved:", OUT_AGREE)
print(f"tokens: in(uncached)={ti} cache_read={tc} out={to} | approx cost ${cost:.4f}")

--- complexity ---
complexity
simple          133
intermediate     95
complex          22

--- technicality ---
technicality
non-technical    209
technical         41

--- sensitivity ---
sensitivity
non-sensitive    199
sensitive         51

remaining missing: {'complexity': 0, 'technicality': 0, 'sensitivity': 0}


saved: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/scaleup_queries_v2_faceted_filled.parquet
saved: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/facet_imputation_log.csv (111 rows)
saved: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/facet_agreement_summary.csv
tokens: in(uncached)=249974 cache_read=0 out=5900 | approx cost $0.8384
